# Hito 2 — Generación de anuncios contextuales con IA generativa

Para cada texto de `muestra_evaluacion` (50 noticias + 50 artículos de WikiHow) se
pide a un modelo generativo (Claude, vía API de Anthropic) que, en dos pasos:

1. asigne una **categoría contextual**: una de las 12 categorías originales del
   corpus (se usan estas, y no una taxonomía publicitaria propia, para poder
   evaluar la clasificación con las mismas etiquetas del Hito 1);
2. genere un **anuncio** breve adecuado a esa categoría y al contenido del texto.

Después, dos anotadores evalúan de forma independiente la categoría y el encaje del
anuncio, y se mide el acuerdo entre ambos.

**Configuración de la API key:** copiar `.env.example` como `.env` en la raíz del
repositorio y escribir ahí la clave (`ANTHROPIC_API_KEY=...`). El archivo `.env`
está en `.gitignore`, por lo que la clave nunca aparece en el notebook ni se sube a
GitHub.

## 1. Configuración

In [ ]:
import os
import re
from pathlib import Path

import anthropic
import pandas as pd
from dotenv import load_dotenv
from sklearn.metrics import cohen_kappa_score
from tqdm.auto import tqdm

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "hito2"

# Carga ANTHROPIC_API_KEY desde el archivo .env (no versionado)
load_dotenv(ROOT / ".env")
if not os.getenv("ANTHROPIC_API_KEY"):
    raise RuntimeError("Falta ANTHROPIC_API_KEY: crea el archivo .env a partir de .env.example")

client = anthropic.Anthropic()  # lee la clave del entorno, nunca escrita aquí
MODELO = "claude-opus-5-5"

In [ ]:
muestra_evaluacion = pd.read_csv(DATA / "muestra_evaluacion.csv")
print(muestra_evaluacion.shape)
muestra_evaluacion["source_type"].value_counts()

## 2. Prompt

Las categorías permitidas son las 12 del corpus del Hito 1. Se pide al modelo un
formato de salida fijo (`CATEGORIA:` / `ANUNCIO:`) para poder extraer los campos
automáticamente.

In [ ]:
CATEGORIAS_PERMITIDAS = {
    "alimentation": "alimentación, gastronomía, recetas y bebidas",
    "astronomy": "astronomía, espacio y ciencias del universo",
    "economy": "economía, finanzas, empresas y empleo",
    "fashion": "moda, belleza y estilo",
    "medicine": "medicina, salud y bienestar",
    "military": "defensa, ejército y conflictos armados",
    "motor": "motor, automóviles y vehículos",
    "play": "ocio, juegos, videojuegos y entretenimiento",
    "politics": "política, gobierno e instituciones",
    "religion": "religión y espiritualidad",
    "sport": "deporte y competiciones deportivas",
    "tech": "tecnología, informática, internet y dispositivos",
}


def crear_prompt(texto):
    lista = "\n".join(f"- {cat}: {desc}" for cat, desc in CATEGORIAS_PERMITIDAS.items())
    return f"""Eres un sistema de publicidad contextual. Vas a leer el texto de una página web.

Tareas:
1. Clasifica el texto en UNA sola categoría de la siguiente lista. Escribe exactamente el identificador (la palabra antes de los dos puntos), aunque el encaje no sea perfecto elige la más cercana.
{lista}
2. Redacta un anuncio breve en español (una o dos frases, máximo 30 palabras) de un producto o servicio adecuado a la categoría asignada y coherente con el contenido concreto del texto.

Responde exactamente con este formato y nada más:
CATEGORIA: <identificador>
ANUNCIO: <texto del anuncio>

Texto:
<<<
{texto}
>>>"""


print(crear_prompt("Texto de ejemplo."))

## 3. Llamada al modelo

`generar_respuesta(texto)` construye el prompt con `crear_prompt`, llama al modelo y
devuelve el texto de la respuesta. Se usa esfuerzo `low` porque la tarea es corta;
`fallbacks="default"` hace que, si el modelo rechazase una petición por sus filtros
de seguridad, la API la reintente automáticamente con otro modelo.

In [ ]:
def generar_respuesta(texto):
    respuesta = client.beta.messages.create(
        model=MODELO,
        max_tokens=2048,
        output_config={"effort": "low"},
        betas=["server-side-fallback-2026-07-01"],
        fallbacks="default",
        messages=[{"role": "user", "content": crear_prompt(texto)}],
    )
    if respuesta.stop_reason == "refusal":
        return ""
    return "".join(b.text for b in respuesta.content if b.type == "text").strip()


def extraer_campos(respuesta):
    """Extrae (categoria_generada, anuncio_generado) del texto devuelto por el modelo."""
    cat = re.search(r"CATEGOR[IÍ]A\s*:\s*(.+)", respuesta, flags=re.IGNORECASE)
    anuncio = re.search(r"ANUNCIO\s*:\s*(.+)", respuesta, flags=re.IGNORECASE | re.DOTALL)
    categoria = cat.group(1).strip().strip("*`\"'.[]<> ").lower() if cat else None
    anuncio = " ".join(anuncio.group(1).split()).strip("*\" ") if anuncio else None
    return categoria, anuncio

### 3.1 Prueba con 1 texto

In [ ]:
texto = muestra_evaluacion.loc[0, "text"]
respuesta = generar_respuesta(texto)
print(respuesta)
print(extraer_campos(respuesta))

### 3.2 Prueba con 10 textos

In [ ]:
prueba_10 = muestra_evaluacion.head(10).copy()
prueba_10["respuesta_modelo"] = [generar_respuesta(t) for t in tqdm(prueba_10["text"])]
prueba_10[["categoria_generada", "anuncio_generado"]] = prueba_10["respuesta_modelo"].apply(
    lambda r: pd.Series(extraer_campos(r))
)
prueba_10[["id", "source_type", "categoria_generada", "anuncio_generado"]]

## 4. Generación para los 100 textos

Las respuestas ya obtenidas (incluidas las 10 de la prueba) se reutilizan para no
repetir llamadas. Si una respuesta no contiene una categoría válida, se vuelve a
pedir hasta 2 veces más.

In [ ]:
respuestas = dict(zip(prueba_10["id"], prueba_10["respuesta_modelo"]))
SALIDA = DATA / "muestra_generada.csv"
if SALIDA.exists():  # reanudar si ya se generó antes
    previa = pd.read_csv(SALIDA)
    respuestas |= dict(zip(previa["id"], previa["respuesta_modelo"].fillna("")))

for fila in tqdm(muestra_evaluacion.itertuples(), total=len(muestra_evaluacion)):
    for _ in range(3):
        if extraer_campos(respuestas.get(fila.id, ""))[0] in CATEGORIAS_PERMITIDAS:
            break
        respuestas[fila.id] = generar_respuesta(fila.text)

muestra_generada = muestra_evaluacion[["id", "text", "source_type"]].copy()
muestra_generada["respuesta_modelo"] = muestra_generada["id"].map(respuestas)
muestra_generada[["categoria_generada", "anuncio_generado"]] = muestra_generada["respuesta_modelo"].apply(
    lambda r: pd.Series(extraer_campos(r))
)
muestra_generada.to_csv(SALIDA, index=False)
muestra_generada[["id", "source_type", "categoria_generada", "anuncio_generado"]].head()

## 5. Validación de categorías

In [ ]:
muestra_generada["categoria_valida"] = muestra_generada["categoria_generada"].isin(CATEGORIAS_PERMITIDAS)
invalidas = muestra_generada.loc[~muestra_generada["categoria_valida"]]
sin_anuncio = muestra_generada["anuncio_generado"].isna().sum()

print(f"Categorías válidas: {muestra_generada['categoria_valida'].sum()}/{len(muestra_generada)}")
print(f"Respuestas sin anuncio: {sin_anuncio}")
assert invalidas.empty, f"Categorías fuera de la lista:\n{invalidas[['id', 'categoria_generada']]}"

In [ ]:
pd.crosstab(muestra_generada["categoria_generada"], muestra_generada["source_type"], margins=True)

## 6. Archivos para la evaluación humana

Se crean dos archivos idénticos e independientes, uno por anotador. Cada persona
rellena **solo el suyo**, sin ver el del otro:

| Columna | Valores |
|---|---|
| `categoria_correcta` | 1 = sí, 0 = no |
| `encaje_anuncio` | 2 = encaja claramente, 1 = encaje parcial, 0 = no encaja |
| `comentario` | opcional |

Los archivos no se sobrescriben si ya existen, para no perder anotaciones.

In [ ]:
plantilla = muestra_generada[["id", "source_type", "text", "categoria_generada", "anuncio_generado"]].copy()
plantilla["categoria_correcta"] = pd.NA
plantilla["encaje_anuncio"] = pd.NA
plantilla["comentario"] = ""

for nombre in ["anotador_1.csv", "anotador_2.csv"]:
    ruta = DATA / nombre
    if ruta.exists():
        print(f"{nombre} ya existe: no se sobrescribe")
    else:
        plantilla.to_csv(ruta, index=False, encoding="utf-8-sig")  # utf-8-sig: tildes correctas en Excel
        print(f"Creado {ruta.relative_to(ROOT)}")

## 7. Evaluación del acuerdo entre anotadores

Ejecutar cuando ambos anotadores hayan terminado. Se calcula, en conjunto y por
fuente (`news` / `wikihow`):

- **Porcentaje de acuerdo** y **Cohen's Kappa** para `categoria_correcta` y
  `encaje_anuncio`. Para el encaje, al ser una escala ordinal, se añade también la
  Kappa ponderada cuadrática (un desacuerdo 0 vs 2 penaliza más que 1 vs 2).
- **Categorías correctas según ambos**: % de textos que ambos anotadores marcan
  como correctos.
- **Encaje medio**: media de las puntuaciones de los dos anotadores (escala 0–2).

In [ ]:
def cargar_anotaciones(nombre):
    """Carga un archivo de anotador. Acepta separador , o ; por si se guardó desde Excel."""
    for encoding in ["utf-8-sig", "latin-1"]:
        try:
            df = pd.read_csv(DATA / nombre, sep=None, engine="python", encoding=encoding)
            break
        except UnicodeDecodeError:
            continue
    df = df[["id", "source_type", "categoria_correcta", "encaje_anuncio", "comentario"]]
    for col, validos in [("categoria_correcta", {0, 1}), ("encaje_anuncio", {0, 1, 2})]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        faltan = df[col].isna().sum()
        fuera = (~df[col].dropna().isin(validos)).sum()
        assert faltan == 0, f"{nombre}: {faltan} filas sin rellenar en {col}"
        assert fuera == 0, f"{nombre}: {fuera} valores no válidos en {col}"
        df[col] = df[col].astype(int)
    return df


a1 = cargar_anotaciones("anotador_1.csv")
a2 = cargar_anotaciones("anotador_2.csv")
anotaciones = a1.merge(a2.drop(columns="source_type"), on="id", suffixes=("_1", "_2"), validate="one_to_one")
assert len(anotaciones) == len(muestra_generada), "Los archivos de anotación no contienen los 100 textos"
anotaciones.head()

In [ ]:
def kappa(x, y, weights=None):
    # Si ambos anotadores usan un único valor idéntico, Kappa no está definida
    if len(set(x) | set(y)) < 2:
        return float("nan")
    return cohen_kappa_score(x, y, weights=weights)


def metricas(df):
    c1, c2 = df["categoria_correcta_1"], df["categoria_correcta_2"]
    e1, e2 = df["encaje_anuncio_1"], df["encaje_anuncio_2"]
    return pd.Series({
        "n_textos": len(df),
        "acuerdo_categoria_pct": (c1 == c2).mean() * 100,
        "kappa_categoria": kappa(c1, c2),
        "acuerdo_encaje_pct": (e1 == e2).mean() * 100,
        "kappa_encaje": kappa(e1, e2),
        "kappa_encaje_ponderada": kappa(e1, e2, weights="quadratic"),
        "categoria_correcta_ambos_pct": ((c1 == 1) & (c2 == 1)).mean() * 100,
        "categoria_correcta_anotador_1_pct": c1.mean() * 100,
        "categoria_correcta_anotador_2_pct": c2.mean() * 100,
        "encaje_medio": ((e1 + e2) / 2).mean(),
        "encaje_medio_anotador_1": e1.mean(),
        "encaje_medio_anotador_2": e2.mean(),
    })


resultados = pd.concat(
    {"total": metricas(anotaciones), **{f: metricas(g) for f, g in anotaciones.groupby("source_type")}},
    axis=1,
).T
resultados.index.name = "fuente"
resultados.round(3)

### 7.1 Comparación `news` vs `wikihow`

In [ ]:
comparacion = resultados.loc[["news", "wikihow"]].T
comparacion["diferencia_news_menos_wikihow"] = comparacion["news"] - comparacion["wikihow"]
comparacion.round(3)

In [ ]:
# Encaje medio por categoría generada y fuente
anotaciones_cat = anotaciones.merge(muestra_generada[["id", "categoria_generada"]], on="id")
anotaciones_cat["encaje_medio"] = (anotaciones_cat["encaje_anuncio_1"] + anotaciones_cat["encaje_anuncio_2"]) / 2
anotaciones_cat.pivot_table(index="categoria_generada", columns="source_type",
                            values="encaje_medio", aggfunc="mean").round(2)

## 8. Guardado de resultados

In [ ]:
resultados.round(4).to_csv(DATA / "resultados_evaluacion.csv")
anotaciones_cat.to_csv(DATA / "anotaciones_combinadas.csv", index=False)
print("Guardados: resultados_evaluacion.csv y anotaciones_combinadas.csv")